# Pricing CANN — Résultats finaux et explicabilité

Notebook **rapide à relancer** : charge les modèles déjà entraînés depuis `models/` (sauvegardés par `03_pricing_cann_exploration.ipynb`) plutôt que de les réentraîner. Si un modèle est absent, il est entraîné automatiquement (pattern load-or-train) — utile la première fois ou après suppression du dossier `models/`.

Contenu : GLM baseline, modèle d'interaction ciblée (résultat retenu), NGBoost, et l'analyse SHAP du résidu appris.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
import joblib
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

Device utilisé : cuda


## Données (rapide, pas besoin de cache — chargement CSV + split < 10s)

In [2]:
from src.pricing.data import build_pricing_dataset, train_valid_test_split, get_severity_subset
from src.pricing.features import build_features
from src.pricing.models import (
    fit_glm_poisson, fit_glm_gamma, predict_frequency, predict_severity,
    fit_ngboost_severity, predict_ngboost_severity,
)

df = build_pricing_dataset()
df = build_features(df)
train, valid, test = train_valid_test_split(df)

train_sev = get_severity_subset(train)
test_sev = get_severity_subset(test)

print(f"Train: {len(train):,} | Valid: {len(valid):,} | Test: {len(test):,}")

Train: 406,807 | Valid: 135,602 | Test: 135,604


## GLM Poisson / Gamma — chargés si disponibles, sinon entraînés

In [3]:
glm_path = MODELS_DIR / "glm_poisson.pkl"
gamma_path = MODELS_DIR / "glm_gamma.pkl"

if glm_path.exists():
    model_glm_train = joblib.load(glm_path)
    print("GLM Poisson chargé.")
else:
    model_glm_train = fit_glm_poisson(train)
    joblib.dump(model_glm_train, glm_path)
    print("GLM Poisson entraîné et sauvegardé.")

if gamma_path.exists():
    model_gamma = joblib.load(gamma_path)
    print("GLM Gamma chargé.")
else:
    model_gamma = fit_glm_gamma(train_sev)
    joblib.dump(model_gamma, gamma_path)
    print("GLM Gamma entraîné et sauvegardé.")

print("Déviance/obs train GLM Poisson :", model_glm_train.deviance / len(train))

GLM Poisson chargé.
GLM Gamma chargé.
Déviance/obs train GLM Poisson : 0.3150812491593767


In [4]:
# Prédictions GLM nécessaires pour le CANN (rapide, toujours recalculé)
train["glm_log_pred"] = np.log(predict_frequency(model_glm_train, train))
valid["glm_log_pred"] = np.log(predict_frequency(model_glm_train, valid))
test["glm_log_pred"] = np.log(predict_frequency(model_glm_train, test))

train["log_mu_glm"] = train["glm_log_pred"] + np.log(train["Exposure"])
valid["log_mu_glm"] = valid["glm_log_pred"] + np.log(valid["Exposure"])
test["log_mu_glm"] = test["glm_log_pred"] + np.log(test["Exposure"])

## Modèle d'interaction ciblée (VehPower/VehAge/VehGas/VehBrand) — résultat retenu

Chargé depuis `models/cann_group_interaction.pt` si disponible. Sinon, entraînement complet (400 epochs, quelques minutes sur GPU).

In [5]:
from src.pricing.cann import GroupInteractionNet, GroupDataset, train_group_interaction, poisson_deviance_loss_v2

group_continuous_cols = ["VehPower_norm", "VehAge_norm", "VehGas_code"]

train_group = GroupDataset(train, group_continuous_cols, "VehBrand_code")
valid_group = GroupDataset(valid, group_continuous_cols, "VehBrand_code")
test_group = GroupDataset(test, group_continuous_cols, "VehBrand_code")

model_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)
cann_path = MODELS_DIR / "cann_group_interaction.pt"

if cann_path.exists():
    model_group.load_state_dict(torch.load(cann_path, map_location=device))
    print("Modèle d'interaction ciblée chargé depuis le disque.")
else:
    train_loader = DataLoader(train_group, batch_size=4096, shuffle=True)
    valid_loader = DataLoader(valid_group, batch_size=4096, shuffle=False)
    optimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)
    model_group, best_valid, best_epoch = train_group_interaction(
        model_group, train_loader, valid_loader, n_epochs=400, optimizer=optimizer, device=device
    )
    torch.save(model_group.state_dict(), cann_path)
    print("Modèle entraîné et sauvegardé.")

Modèle d'interaction ciblée chargé depuis le disque.


In [6]:
# Évaluation sur test
test_loader = DataLoader(test_group, batch_size=4096, shuffle=False)

model_group.eval()
test_losses = []
with torch.no_grad():
    for batch in test_loader:
        continuous, brand_code = batch["continuous"].to(device), batch["brand_code"].to(device)
        log_mu_glm, claim_nb = batch["log_mu_glm"].to(device), batch["claim_nb"].to(device)
        log_lambda = model_group(continuous, brand_code, log_mu_glm)
        test_losses.append(poisson_deviance_loss_v2(log_lambda, claim_nb).item())

test_loss_group = sum(test_losses) / len(test_losses)
print(f"Loss sur test (modèle d'interaction) : {test_loss_group:.4f}")
print(f"Référence GLM sur test               : 0.3179")
print(f"Gain relatif : {(test_loss_group / 0.3179 - 1) * 100:.2f}%")

Loss sur test (modèle d'interaction) : 0.3130
Référence GLM sur test               : 0.3179
Gain relatif : -1.56%


## NGBoost — sévérité distributionnelle

In [7]:
ngboost_path = MODELS_DIR / "ngboost_severity.pkl"

if ngboost_path.exists():
    model_ngboost = joblib.load(ngboost_path)
    print("NGBoost chargé.")
else:
    model_ngboost = fit_ngboost_severity(train_sev, n_estimators=300)
    joblib.dump(model_ngboost, ngboost_path)
    print("NGBoost entraîné et sauvegardé.")

ngboost_preds = predict_ngboost_severity(model_ngboost, test_sev)
covered = (
    (test_sev["ClaimAmount_capped"].values >= ngboost_preds["pred_lower_90"].values) &
    (test_sev["ClaimAmount_capped"].values <= ngboost_preds["pred_upper_90"].values)
)
print(f"Couverture empirique intervalle 90% : {covered.mean():.2%}")

NGBoost chargé.
Couverture empirique intervalle 90% : 90.58%


## Explicabilité SHAP du modèle d'interaction ciblée

Explique le résidu appris par le réseau (correction par rapport au GLM), en fonction des variables continues et de l'embedding appris de `VehBrand`.

In [8]:
import shap
from src.pricing.cann import get_shap_input_matrix, ResidualMLPWrapper

X_test_shap = get_shap_input_matrix(model_group, test, group_continuous_cols, "VehBrand_code", device)

np.random.seed(123)
background_idx = np.random.choice(len(X_test_shap), 100, replace=False)
background = torch.tensor(X_test_shap[background_idx], dtype=torch.float32).to(device)

wrapper = ResidualMLPWrapper(model_group).to(device)
wrapper.eval()

explainer = shap.DeepExplainer(wrapper, background)

sample_idx = np.random.choice(len(X_test_shap), 500, replace=False)
X_sample = torch.tensor(X_test_shap[sample_idx], dtype=torch.float32).to(device)

shap_values = explainer.shap_values(X_sample)
feature_names = group_continuous_cols + ["VehBrand_emb_0", "VehBrand_emb_1"]
print("Shape des valeurs SHAP :", np.array(shap_values).shape)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\marie\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\marie\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\marie\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\marie\AppData\Local\Programs\Python\Python310\lib\site-packages\traitlets\config

AttributeError: _ARRAY_API not found

Shape des valeurs SHAP : (500, 5, 1)


In [9]:
shap_array = np.array(shap_values).squeeze()
mean_abs_shap = np.abs(shap_array).mean(axis=0)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

vehbrand_importance = mean_abs_shap[3] + mean_abs_shap[4]
print(f"\nImportance agrégée VehBrand (2 dims d'embedding) : {vehbrand_importance:.6f}")

          feature  mean_abs_shap
1     VehAge_norm       0.171870
3  VehBrand_emb_0       0.133653
2     VehGas_code       0.097082
4  VehBrand_emb_1       0.063735
0   VehPower_norm       0.050153

Importance agrégée VehBrand (2 dims d'embedding) : 0.197389


## Résumé final

| Modèle | Déviance/obs test | vs GLM |
|---|---|---|
| GLM Poisson (référence) | 0.3179 | — |
| CANN générique | ~0.3097\* | pire |
| Interaction ciblée (retenue) | ~0.3129 | **-1.61%** |

\* non recalculé ici, voir `03_pricing_cann_exploration.ipynb` pour la trace complète.